In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io

from scipy.integrate import trapezoid
from scipy.optimize import curve_fit

from calcium_analysis.peaks import (
    get_peak_positions_and_properties,
    get_timeseries_per_spike_df,
    append_segment_bounds_using_relative_prominence,
    append_segment_bounds_using_local_minima,
)

from calcium_analysis.smoothing import (
    rolling_quantile,
    rolling_gaussian_mean,
)
from calcium_analysis.fitting import fit_exponential_decay_per_spike

# Read data from excel and convert to "standard" format

In [2]:
fps = 11
DT = 1 / fps

In [3]:
df = pd.read_excel("/Users/mariaclaudia/Downloads/test_scOscillations.xlsx")
df

,Row,Column,Timepoint,Object ID,Tracked Loaded Population - Intensity Cell th/Fluo-4 Mean,Cell Type,Compound,Concentration
0,2,2,1,25,554.125,c1,LTX1,0
1,2,2,2,25,563.279,c1,LTX1,0
2,2,2,3,25,557.382,c1,LTX1,0
3,2,2,4,25,551.346,c1,LTX1,0
4,2,2,5,25,557.500,c1,LTX1,0
...,...,...,...,...,...,...,...,...
985,2,3,326,38,325.787,g1,LTX1,0
986,2,3,327,38,321.840,g1,LTX1,0
987,2,3,328,38,322.027,g1,LTX1,0
988,2,3,329,38,328.080,g1,LTX1,0


In [4]:
time = df["Timepoint"] * DT
df["Timepoint"] = time
df = df.rename(
    columns={
        "Timepoint": "time",
        "Tracked Loaded Population - Intensity Cell th/Fluo-4 Mean": "value",
    }
)
df = df.set_index(["Row", "Column", "Object ID", "time"])

In [5]:
df

value Cell Type Compound  Concentration
Row Column Object ID time                                                
2   2      25        0.090909   554.125        c1     LTX1              0
                     0.181818   563.279        c1     LTX1              0
                     0.272727   557.382        c1     LTX1              0
                     0.363636   551.346        c1     LTX1              0
                     0.454545   557.500        c1     LTX1              0
...                                 ...       ...      ...            ...
    3      38        29.636364  325.787        g1     LTX1              0
                     29.727273  321.840        g1     LTX1              0
                     29.818182  322.027        g1     LTX1              0
                     29.909091  328.080        g1     LTX1              0
                     30.000000  328.180        g1     LTX1              0

[990 rows x 4 columns]

# Subract baseline, normalize, and smooth

In [12]:
baseline = rolling_quantile(df["value"], quantile=0.10, window_size=100, min_periods=50)

normalized_value = df["value"] / baseline - 1

smoothed_normalized = rolling_gaussian_mean(
    normalized_value,
    kernel_width=6,
    kernel_sigma=2,
)

# Compile a Dataframe with various properties for each peak

In [7]:
peaks_and_properties = get_peak_positions_and_properties(
    smoothed_normalized,
    height_z_score_threshold=3,
    prominence_threshold_over_sigma=2,
    min_delta_t=2,
    rel_prominences_for_widths=[0.5, 0.75],
)


metadata_to_add = df.groupby(level=["Row", "Column", "Object ID"])[
    ["Compound", "Concentration", "Cell Type"]
].first()
peaks_and_properties = peaks_and_properties.merge(
    metadata_to_add, left_index=True, right_index=True, how="left"
)

In [8]:
peaks_and_properties

peak_heights  prominences  left_bases  \
Row Column Object ID peak_index                                          
2   2      1         0               0.057595     0.056769         245   
    3      38        0               0.419358     0.417086         129   
                     1               0.390087     0.387554         222   

                                 right_bases  peak_centers_seconds  \
Row Column Object ID peak_index                                      
2   2      1         0                   329             24.000000   
    3      38        0                   311             13.000000   
                     1                   311             22.818182   

                                 peak_centers_idx   width_50  \
Row Column Object ID peak_index                                
2   2      1         0                        263  36.850113   
    3      38        0                        142  14.415614   
                     1                        250  31.230196   

                                 width_50_start_idx  width_50_end_idx  \
Row Column Object ID peak_index                                         
2   2      1         0                          253               290   
    3      38        0                          137               152   
                     1                          229               260   

                                 width_50_start_time  width_50_end_time  \
Row Column Object ID peak_index                                           
2   2      1         0                     23.090909          26.454545   
    3      38        0                     12.545455          13.909091   
                     1                     20.909091          23.727273   

                                  width_75  width_75_start_idx  \
Row Column Object ID peak_index                                  
2   2      1         0           47.358385                 252   
    3      38        0           24.880521                 136   
                     1           40.721277                 228   

                                 width_75_end_idx  width_75_start_time  \
Row Column Object ID peak_index                                          
2   2      1         0                        299            23.000000   
    3      38        0                        161            12.454545   
                     1                        268            20.818182   

                                 width_75_end_time Compound  Concentration  \
Row Column Object ID peak_index                                              
2   2      1         0                   27.272727     LTX1              0   
    3      38        0                   14.727273     LTX1              0   
                     1                   24.454545     LTX1              0   

                                Cell Type  
Row Column Object ID peak_index            
2   2      1         0                 c1  
    3      38        0                 g1  
                     1                 g1

# Compile a DataFrame with the signal segment corresponding to each peak

## Option 1: using relative prominence to determine the segment boundary

In [9]:
peaks_and_properties = append_segment_bounds_using_relative_prominence(
    peaks_and_properties,
    smoothed_normalized,
    rel_prominence=0.75,
)
peaks_timeseries = get_timeseries_per_spike_df(
    smoothed_normalized, peaks_and_properties
)

In [10]:
peaks_timeseries

signal_segment
Row Column Object ID peak_index time_from_peak                
2   2      1         0          -1.000000             0.014855
                                -0.909091             0.026069
                                -0.818182             0.036152
                                -0.727273             0.045000
                                -0.636364             0.046350
...                                                        ...
    3      38        1           1.272727             0.160318
                                 1.363636             0.147197
                                 1.454545             0.132975
                                 1.545455             0.119075
                                 1.636364             0.107457

[115 rows x 1 columns]

In [11]:
peaks_and_properties

peak_heights  prominences  left_bases  \
Row Column Object ID peak_index                                          
2   2      1         0               0.057595     0.056769         245   
    3      38        0               0.419358     0.417086         129   
                     1               0.390087     0.387554         222   

                                 right_bases  peak_centers_seconds  \
Row Column Object ID peak_index                                      
2   2      1         0                   329             24.000000   
    3      38        0                   311             13.000000   
                     1                   311             22.818182   

                                 peak_centers_idx   width_50  \
Row Column Object ID peak_index                                
2   2      1         0                        263  36.850113   
    3      38        0                        142  14.415614   
                     1                        250  31.230196   

                                 width_50_start_idx  width_50_end_idx  \
Row Column Object ID peak_index                                         
2   2      1         0                          253               290   
    3      38        0                          137               152   
                     1                          229               260   

                                 width_50_start_time  ...  width_75_start_idx  \
Row Column Object ID peak_index                       ...                       
2   2      1         0                     23.090909  ...                 252   
    3      38        0                     12.545455  ...                 136   
                     1                     20.909091  ...                 228   

                                 width_75_end_idx  width_75_start_time  \
Row Column Object ID peak_index                                          
2   2      1         0                        299            23.000000   
    3      38        0                        161            12.454545   
                     1                        268            20.818182   

                                 width_75_end_time  Compound  Concentration  \
Row Column Object ID peak_index                                               
2   2      1         0                   27.272727      LTX1              0   
    3      38        0                   14.727273      LTX1              0   
                     1                   24.454545      LTX1              0   

                                Cell Type  segment_start_idx segment_end_idx  \
Row Column Object ID peak_index                                                
2   2      1         0                 c1                252             299   
    3      38        0                 g1                136             161   
                     1                 g1                228             268   

                                 segment_truncated  
Row Column Object ID peak_index                     
2   2      1         0                       False  
    3      38        0                       False  
                     1                       False  

[3 rows x 22 columns]

## Option 2: using local minima to determine the segment boundary

In [11]:
peaks_and_properties = append_segment_bounds_using_local_minima(
    peaks_and_properties,
    smoothed_normalized,
)
peaks_timeseries_alt = get_timeseries_per_spike_df(
    smoothed_normalized, peaks_and_properties
)

In [12]:
peaks_timeseries_alt

signal_segment
Row Column Object ID peak_index time_from_peak                
2   2      1         0          -0.363636             0.039636
                                -0.272727             0.043654
                                -0.181818             0.048027
                                -0.090909             0.053529
                                 0.000000             0.057595
...                                                        ...
    3      38        1           3.272727             0.024370
                                 3.363636             0.021830
                                 3.454545             0.019387
                                 3.545455             0.019018
                                 3.636364             0.017649

[108 rows x 1 columns]

# Fit each peak decay as an exponential decay

In [13]:
decay_data = fit_exponential_decay_per_spike(
    peaks_timeseries_alt["signal_segment"]
).where(~peaks_and_properties["segment_truncated"])

In [14]:
decay_data

peak_over_baseline       tau  baseline  \
Row Column Object ID peak_index                                           
2   2      1         0                          NaN       NaN       NaN   
    3      38        0                     0.464395  1.345064 -0.014313   
                     1                     0.439225  1.435872 -0.024544   

                                 mean_square_error        r2  
Row Column Object ID peak_index                               
2   2      1         0                         NaN       NaN  
    3      38        0                    0.000071  0.995109  
                     1                    0.000040  0.996962

In [15]:
## Frequency
total_rec_time = (
    smoothed_normalized.groupby(["Row", "Column", "Object ID"]).count() * DT
)

freq = (peaks_and_properties.groupby(["Row", "Column", "Object ID"]).size() / total_rec_time)

# Nan correspond to no peaks -> frequency=0
freq = freq.fillna(0)

freq = freq.rename("Peak Frequency [Hz]")

In [16]:
freq

Row  Column  Object ID
2    2       1            0.033333
             25           0.000000
     3       38           0.066667
Name: Peak Frequency [Hz], dtype: float64

# Extracting a new property from the timeseries

## Example: AUC

In [17]:
from calcium_analysis.multiindex_decorators import (
    support_multiindex_signal_single_row_returns,
)


@support_multiindex_signal_single_row_returns(time_name="time_from_peak")
def auc_per_series(signal: pd.Series) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "AUC": trapezoid(signal.values, dx=DT),
            }
        ]
    )


auc = auc_per_series(peaks_timeseries_alt["signal_segment"])
auc

AUC
Row Column Object ID peak_index          
2   2      1         0           0.051381
    3      38        0           0.681489
                     1           0.654439